# KELT-20b Transmission Spectral Diagnostics

This notebook is a standalone diagnostic for the prepared KELT-20b high-resolution transmission spectra on disk. It does not reprocess raw data or change pipeline products. The goal is to inspect what the retrieval-ready time-series bundles look like, compare the two available transit epochs, and zoom into individual line windows in a phase-aware way.

## Config

The defaults point at the prepared KELT-20b transmission bundles currently present under `input/hrs/transmission/kelt20b/`. Set `EPOCHS` or `ARMS` smaller while iterating if the notebook feels too chatty.

In [ ]:
PLANET = "KELT-20b"
PLANET_SLUG = "kelt20b"
EPHEMERIS = "Duck24"

EPOCHS = ("20190504", "20250601")
ARMS = ("blue", "red")

# Plot readability knobs.
PHASE_ROWS_TOP_TO_BOTTOM = True
MATRIX_PERCENTILE = 99.5
PROFILE_MAX_BINS = 1400
OBSERVED_MAX_BINS = 750
COLLAPSED_MAX_BINS = 900
MASK_SIGMA_THRESHOLD = 0.5  # Prep uses sigma=1 as an ignore/downweight marker.

# Transmission line microscope knobs.
LINE_WINDOW_KMS = 150.0
LINE_BIN_KMS = 2.0
INCLUDE_SYSTEMIC_IN_PLANET_TRAIL = False

# Set to an integer, e.g. 1, if you only want the first epoch while iterating.
MAX_EPOCHS_TO_PLOT = None

In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config
from config_utils import get_params
from dataio.collapse_transmission_timeseries_to_1d import compute_contact_phases

DATA_ROOT = REPO_ROOT / "input" / "hrs" / "transmission" / PLANET_SLUG
LRS_ROOT = REPO_ROOT / "input" / "lrs" / "transmission" / PLANET_SLUG
PHOT_ROOT = REPO_ROOT / "input" / "phot" / "transmission" / PLANET_SLUG

planet_cfg = get_params(PLANET, EPHEMERIS)
contacts = compute_contact_phases(planet_cfg)
CKMS = 299792.458

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
})

DATA_ROOT

## Helpers

In [ ]:
def _load_optional_array(path: Path):
    return np.load(path, allow_pickle=True) if path.exists() else None


def load_bundle(epoch: str, arm: str) -> dict:
    path = DATA_ROOT / epoch / arm
    required = ("wavelength.npy", "data.npy", "sigma.npy", "phase.npy")
    missing = [name for name in required if not (path / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing {missing} in {path}")

    meta_path = path / "timeseries_prep.json"
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    bundle = {
        "epoch": epoch,
        "arm": arm,
        "path": path,
        "wavelength": np.asarray(np.load(path / "wavelength.npy"), dtype=float),
        "data": np.asarray(np.load(path / "data.npy"), dtype=float),
        "sigma": np.asarray(np.load(path / "sigma.npy"), dtype=float),
        "phase": np.asarray(np.load(path / "phase.npy"), dtype=float),
        "jd": _load_optional_array(path / "jd.npy"),
        "airmass": _load_optional_array(path / "airmass.npy"),
        "snr": _load_optional_array(path / "snr.npy"),
        "exptime": _load_optional_array(path / "exptime.npy"),
        "pre_sysrem_data": _load_optional_array(path / "pre_sysrem_data.npy"),
        "pre_sysrem_sigma": _load_optional_array(path / "pre_sysrem_sigma.npy"),
        "meta": meta,
    }
    collapsed_names = ("wavelength_transmission.npy", "spectrum_transmission.npy", "uncertainty_transmission.npy")
    if all((path / name).exists() for name in collapsed_names):
        bundle["collapsed"] = {
            "wavelength": np.asarray(np.load(path / "wavelength_transmission.npy"), dtype=float),
            "spectrum": np.asarray(np.load(path / "spectrum_transmission.npy"), dtype=float),
            "uncertainty": np.asarray(np.load(path / "uncertainty_transmission.npy"), dtype=float),
        }
    else:
        bundle["collapsed"] = None

    sysrem_path = path / "U_sysrem.npz"
    bundle["sysrem"] = np.load(sysrem_path, allow_pickle=True) if sysrem_path.exists() else None
    return bundle


def load_all_bundles() -> list[dict]:
    records = []
    for epoch in EPOCHS:
        for arm in ARMS:
            path = DATA_ROOT / epoch / arm
            if path.exists():
                records.append(load_bundle(epoch, arm))
            else:
                print(f"Skipping missing bundle: {path}")
    return records


bundles = load_all_bundles()
bundles_by_key = {(b["epoch"], b["arm"]): b for b in bundles}

In [ ]:
def finite_good(data, sigma=None):
    mask = np.isfinite(data)
    if sigma is not None:
        mask &= np.isfinite(sigma) & (sigma > 0) & (sigma < MASK_SIGMA_THRESHOLD)
    return mask


def robust_limits(values, percentile=MATRIX_PERCENTILE, symmetric=True, floor=None):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None
    if symmetric:
        vmax = float(np.nanpercentile(np.abs(arr), percentile))
        if floor is not None:
            vmax = max(vmax, floor)
        if not np.isfinite(vmax) or vmax <= 0:
            return None
        return -vmax, vmax
    lo = float(np.nanpercentile(arr, 100.0 - percentile))
    hi = float(np.nanpercentile(arr, percentile))
    if floor is not None and hi - lo < floor:
        mid = 0.5 * (lo + hi)
        lo, hi = mid - floor / 2, mid + floor / 2
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return None
    return lo, hi


def display_wavelength_nm(wave_angstrom):
    return np.asarray(wave_angstrom, dtype=float) / 10.0


def phase_order(bundle):
    order = np.argsort(bundle["phase"])
    if not PHASE_ROWS_TOP_TO_BOTTOM:
        order = order[::-1]
    return order


def phase_bin_label(phase):
    p = float(phase)
    if contacts["T2"] <= p <= contacts["T3"]:
        return "full transit"
    if contacts["T1"] <= p < contacts["T2"]:
        return "ingress"
    if contacts["T3"] < p <= contacts["T4"]:
        return "egress"
    return "out"


def gap_aware_bins(wave, max_bins=700, gap_factor=8.0):
    wave = np.asarray(wave, dtype=float)
    good = np.flatnonzero(np.isfinite(wave))
    if good.size == 0:
        return []
    diffs = np.diff(wave[good])
    finite_diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
    if finite_diffs.size:
        gap_limit = gap_factor * np.nanmedian(finite_diffs)
        split_at = np.flatnonzero(diffs > gap_limit) + 1
        groups = np.split(good, split_at)
    else:
        groups = [good]
    bins = []
    for group in groups:
        if group.size == 0:
            continue
        nbin = max(1, int(math.ceil(group.size / max_bins)))
        bins.extend([chunk for chunk in np.array_split(group, max(1, int(math.ceil(group.size / nbin)))) if chunk.size])
    return bins


def binned_series(wave, values, errors=None, max_bins=700):
    wave = np.asarray(wave, dtype=float)
    values = np.asarray(values, dtype=float)
    errors = None if errors is None else np.asarray(errors, dtype=float)
    wb, vb, eb = [], [], []
    for idx in gap_aware_bins(wave, max_bins=max_bins):
        w = wave[idx]
        v = values[idx]
        if errors is not None:
            e = errors[idx]
            good = np.isfinite(w) & np.isfinite(v) & np.isfinite(e) & (e > 0) & (e < MASK_SIGMA_THRESHOLD)
            if np.any(good):
                weight = 1.0 / np.square(e[good])
                wb.append(float(np.average(w[good], weights=weight)))
                vb.append(float(np.average(v[good], weights=weight)))
                eb.append(float(np.sqrt(1.0 / np.sum(weight))))
                continue
        good = np.isfinite(w) & np.isfinite(v)
        if np.any(good):
            wb.append(float(np.nanmean(w[good])))
            vb.append(float(np.nanmean(v[good])))
            eb.append(float(np.nanstd(v[good]) / max(1.0, np.sqrt(np.count_nonzero(good)))))
    return np.asarray(wb), np.asarray(vb), np.asarray(eb)


def column_stats(bundle):
    data = np.asarray(bundle["data"], dtype=float)
    sigma = np.asarray(bundle["sigma"], dtype=float)
    good = finite_good(data, sigma)
    weight = np.where(good, 1.0 / np.square(sigma), 0.0)
    wsum = np.sum(weight, axis=0)
    weighted_mean = np.divide(
        np.sum(np.where(good, data * weight, 0.0), axis=0),
        wsum,
        out=np.full(data.shape[1], np.nan),
        where=wsum > 0,
    )
    err_mean = np.divide(1.0, np.sqrt(wsum), out=np.full(data.shape[1], np.nan), where=wsum > 0)
    typical_sigma = np.sqrt(np.nanmean(np.where(good, np.square(sigma), np.nan), axis=0))
    count = np.sum(good, axis=0)
    scatter = np.nanstd(np.where(good, data, np.nan), axis=0)
    scatter = np.where(count >= 2, scatter, np.nan)
    return {
        "mean": weighted_mean,
        "err_mean": err_mean,
        "typical_sigma": typical_sigma,
        "scatter": scatter,
        "count": count,
    }


def planet_velocity_kms(phase):
    rv = float(planet_cfg.get("Kp", 0.0)) * np.sin(2.0 * np.pi * np.asarray(phase, dtype=float))
    if INCLUDE_SYSTEMIC_IN_PLANET_TRAIL:
        rv = rv + float(planet_cfg.get("RV_abs", 0.0))
    return rv

In [ ]:
def shade_contacts(ax, orientation="vertical", label=False):
    # Phase contacts are only meaningful on phase axes.
    if orientation == "horizontal":
        ax.axhspan(contacts["T1"], contacts["T4"], color="tab:green", alpha=0.06, label="T1-T4" if label else None)
        ax.axhspan(contacts["T2"], contacts["T3"], color="tab:green", alpha=0.10, label="T2-T3" if label else None)
        for name, phase in contacts.items():
            ax.axhline(phase, color="0.35", lw=0.7, ls=":" )
    else:
        ax.axvspan(contacts["T1"], contacts["T4"], color="tab:green", alpha=0.06, label="T1-T4" if label else None)
        ax.axvspan(contacts["T2"], contacts["T3"], color="tab:green", alpha=0.10, label="T2-T3" if label else None)
        for name, phase in contacts.items():
            ax.axvline(phase, color="0.35", lw=0.7, ls=":" )


def plot_matrix(bundle, field="data", title=None, symmetric=True, percentile=MATRIX_PERCENTILE):
    wave_nm = display_wavelength_nm(bundle["wavelength"])
    order = phase_order(bundle)
    phase = bundle["phase"][order]
    matrix = np.asarray(bundle[field], dtype=float)[order]
    if field == "sigma":
        matrix = np.where(np.isfinite(matrix) & (matrix < MASK_SIGMA_THRESHOLD), matrix, np.nan)
        symmetric = False
    limits = robust_limits(matrix, percentile=percentile, symmetric=symmetric)
    fig, ax = plt.subplots(figsize=(12, 4.4), constrained_layout=True)
    im = ax.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r" if symmetric else "viridis",
        vmin=None if limits is None else limits[0],
        vmax=None if limits is None else limits[1],
        extent=[float(np.nanmin(wave_nm)), float(np.nanmax(wave_nm)), len(order) - 0.5, -0.5],
    )
    ticks = np.arange(len(order))
    if len(ticks) > 14:
        keep = np.unique(np.linspace(0, len(ticks) - 1, 12).astype(int))
    else:
        keep = ticks
    ax.set_yticks(keep)
    ax.set_yticklabels([f"{phase[i]:+.4f}" for i in keep])
    ax.set_ylabel("Orbital phase")
    ax.set_xlabel("Wavelength [nm]")
    ax.set_title(title or f"{bundle['epoch']} {bundle['arm']} {field} rows sorted by phase")
    label = "Residual flux" if field == "data" else "Uncertainty"
    cb = fig.colorbar(im, ax=ax, pad=0.012, fraction=0.035)
    cb.set_label(label)
    return fig, ax


def plot_profiles(bundle):
    stats = column_stats(bundle)
    wave = bundle["wavelength"]
    wb, mean_b, err_b = binned_series(wave, stats["mean"], stats["err_mean"], max_bins=PROFILE_MAX_BINS)
    _, sig_b, _ = binned_series(wave, stats["typical_sigma"], None, max_bins=PROFILE_MAX_BINS)
    _, scatter_b, _ = binned_series(wave, stats["scatter"], None, max_bins=PROFILE_MAX_BINS)
    fig, axes = plt.subplots(3, 1, figsize=(12, 7.0), sharex=True, constrained_layout=True, gridspec_kw={"hspace": 0.08})
    axes[0].plot(display_wavelength_nm(wb), mean_b, lw=0.9, color="tab:blue", label="weighted mean residual")
    axes[0].fill_between(display_wavelength_nm(wb), mean_b - err_b, mean_b + err_b, color="tab:blue", alpha=0.16, lw=0, label="uncertainty on mean")
    axes[1].plot(display_wavelength_nm(wb), sig_b, lw=0.9, color="tab:orange", label="typical per-exposure sigma")
    axes[2].plot(display_wavelength_nm(wb), scatter_b, lw=0.9, color="0.25", label="row scatter")
    axes[0].axhline(0, color="0.35", lw=0.8)
    for ax in axes:
        ax.legend(loc="upper right")
    lim = robust_limits(mean_b, percentile=99.2, symmetric=True, floor=2e-5)
    if lim:
        axes[0].set_ylim(*lim)
    for ax, arr in [(axes[1], sig_b), (axes[2], scatter_b)]:
        lim = robust_limits(arr, percentile=99.0, symmetric=False)
        if lim:
            ax.set_ylim(*lim)
    axes[0].set_ylabel("Residual flux")
    axes[1].set_ylabel("Uncertainty")
    axes[2].set_ylabel("Row scatter")
    axes[2].set_xlabel("Wavelength [nm]")
    axes[0].set_title(f"Bundle profiles: {bundle['epoch']} {bundle['arm']}")
    return fig, axes


def plot_observed_summary(bundle):
    stats = column_stats(bundle)
    wb, vb, eb = binned_series(bundle["wavelength"], stats["mean"], stats["err_mean"], max_bins=OBSERVED_MAX_BINS)
    fig, ax = plt.subplots(figsize=(12, 4.4), constrained_layout=True)
    ax.errorbar(display_wavelength_nm(wb), vb, yerr=eb, fmt="o", ms=2.0, lw=0.55, color="k", ecolor="0.35", alpha=0.85, label="weighted mean bins")
    ax.axhline(0, color="0.35", lw=0.8, ls="--")
    lim = robust_limits(vb, percentile=99.0, symmetric=True, floor=3e-5)
    if lim:
        ax.set_ylim(*lim)
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Processed residual flux")
    ax.set_title(f"Observed residual summary: {bundle['epoch']} {bundle['arm']}")
    ax.legend(loc="upper right")
    return fig, ax


def plot_collapsed_transmission(bundle):
    collapsed = bundle.get("collapsed")
    if collapsed is None:
        print(f"No collapsed wavelength/spectrum/uncertainty transmission product for {bundle['epoch']} {bundle['arm']}.")
        return None
    wave = collapsed["wavelength"]
    spectrum = collapsed["spectrum"] - 1.0
    unc = collapsed["uncertainty"]
    wb, sb, eb = binned_series(wave, spectrum, unc, max_bins=COLLAPSED_MAX_BINS)
    fig, ax = plt.subplots(figsize=(12, 4.4), constrained_layout=True)
    ax.errorbar(display_wavelength_nm(wb), sb, yerr=eb, fmt="o", ms=2.0, lw=0.55, color="k", ecolor="0.35", alpha=0.85)
    ax.axhline(0, color="0.35", lw=0.8, ls="--")
    lim = robust_limits(sb, percentile=99.0, symmetric=True, floor=2e-4)
    if lim:
        ax.set_ylim(*lim)
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Transmission - 1")
    ax.set_title(f"Collapsed transmission product: {bundle['epoch']} {bundle['arm']}")
    return fig, ax

## Inventory And Metadata

In [ ]:
def bundle_summary_rows(bundles):
    rows = []
    for b in bundles:
        meta = b["meta"]
        phase = b["phase"]
        data = b["data"]
        sigma = b["sigma"]
        good = finite_good(data, sigma)
        rows.append({
            "epoch": b["epoch"],
            "arm": b["arm"],
            "n_exp": data.shape[0],
            "n_wave": data.shape[1],
            "wave_min_nm": float(np.nanmin(b["wavelength"]) / 10.0),
            "wave_max_nm": float(np.nanmax(b["wavelength"]) / 10.0),
            "phase_min": float(np.nanmin(phase)),
            "phase_max": float(np.nanmax(phase)),
            "in_T1_T4": int(np.count_nonzero((phase >= contacts["T1"]) & (phase <= contacts["T4"]))),
            "in_T2_T3": int(np.count_nonzero((phase >= contacts["T2"]) & (phase <= contacts["T3"]))),
            "run_sysrem": meta.get("run_sysrem"),
            "doppler_shadow": meta.get("doppler_shadow_applied"),
            "shadow_scale": meta.get("doppler_shadow_scaling"),
            "valid_fraction": float(np.count_nonzero(good) / good.size),
            "has_pre_sysrem": b["pre_sysrem_data"] is not None,
            "has_collapsed": b["collapsed"] is not None,
            "path": str(b["path"].relative_to(REPO_ROOT)),
        })
    return rows


summary = pd.DataFrame(bundle_summary_rows(bundles))
display(summary)
print("Contact phases:", {k: round(v, 6) for k, v in contacts.items()})

In [ ]:
def plot_metadata_epoch(epoch, epoch_bundles):
    fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True, constrained_layout=True, gridspec_kw={"hspace": 0.08})
    for b in epoch_bundles:
        phase = np.asarray(b["phase"], dtype=float)
        order = np.argsort(phase)
        label = b["arm"]
        x = np.arange(phase.size)
        axes[0].plot(x, phase[order], marker="o", ms=3, lw=0.8, label=label)
        if b["airmass"] is not None:
            axes[1].plot(x, np.asarray(b["airmass"], dtype=float)[order], marker="o", ms=3, lw=0.8, label=label)
        if b["snr"] is not None:
            axes[2].plot(x, np.asarray(b["snr"], dtype=float)[order], marker="o", ms=3, lw=0.8, label=label)
        if b["exptime"] is not None:
            axes[3].plot(x, np.asarray(b["exptime"], dtype=float)[order], marker="o", ms=3, lw=0.8, label=label)
    shade_contacts(axes[0], orientation="horizontal", label=True)
    axes[0].set_ylabel("Phase")
    axes[1].set_ylabel("Airmass")
    axes[2].set_ylabel("SNR")
    axes[3].set_ylabel("Exp. time")
    axes[3].set_xlabel("Exposure index after phase sort")
    axes[0].set_title(f"Exposure metadata sorted by phase: {epoch}")
    for ax in axes:
        ax.legend(loc="best")
    return fig, axes


epochs_to_plot = EPOCHS if MAX_EPOCHS_TO_PLOT is None else EPOCHS[:MAX_EPOCHS_TO_PLOT]
for epoch in epochs_to_plot:
    epoch_bundles = [b for b in bundles if b["epoch"] == epoch]
    if epoch_bundles:
        plot_metadata_epoch(epoch, epoch_bundles)
        plt.show()

## Phase-Ordered Time-Series Matrices

Rows are sorted by orbital phase. The y-axis is phase, not date or file index. The data matrices show processed residual flux; sigma matrices hide `sigma >= 0.5` ignore pixels so they do not dominate the color scale.

In [ ]:
plot_bundles = bundles if MAX_EPOCHS_TO_PLOT is None else [b for b in bundles if b["epoch"] in EPOCHS[:MAX_EPOCHS_TO_PLOT]]
for b in plot_bundles:
    plot_matrix(b, "data", title=f"Transmission residual rows: {b['epoch']} {b['arm']}")
    plot_matrix(b, "sigma", title=f"Transmission uncertainty rows: {b['epoch']} {b['arm']}")
    plt.show()

## Per-Bundle Profiles

These panels keep three different quantities separate: weighted residual mean, typical per-exposure sigma, and row-to-row scatter. The blue uncertainty band on the first panel is the uncertainty on the mean, not the raw per-exposure sigma.

In [ ]:
for b in plot_bundles:
    plot_profiles(b)
    plt.show()

## Observed Residual Summary

This is the broad collapsed diagnostic for each time-series bundle. It is still only a diagnostic; the retrieval consumes the exposure-level `data.npy` and `sigma.npy` products, not these displayed binned points.

In [ ]:
for b in plot_bundles:
    plot_observed_summary(b)
    plt.show()

## Collapsed Transmission Products On Disk

Only bundles with `wavelength_transmission.npy`, `spectrum_transmission.npy`, and `uncertainty_transmission.npy` appear here. In the current tree, these are the red-arm collapsed products.

In [ ]:
for b in plot_bundles:
    plot_collapsed_transmission(b)
    plt.show()

## Epoch Comparison

In [ ]:
def compare_observed_by_arm(bundles):
    fig, axes = plt.subplots(len(ARMS), 1, figsize=(12, 3.4 * len(ARMS)), sharex=False, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, arm in zip(axes, ARMS):
        for b in [r for r in bundles if r["arm"] == arm]:
            stats = column_stats(b)
            wb, vb, eb = binned_series(b["wavelength"], stats["mean"], stats["err_mean"], max_bins=OBSERVED_MAX_BINS)
            ax.plot(display_wavelength_nm(wb), vb, lw=0.9, alpha=0.8, label=f"{b['epoch']} phi {np.nanmin(b['phase']):+.3f}..{np.nanmax(b['phase']):+.3f}")
        ax.axhline(0, color="0.35", lw=0.8, ls="--")
        ax.set_ylabel("Residual flux")
        ax.set_title(f"Weighted residual mean comparison: {arm}")
        ax.legend(loc="best")
    axes[-1].set_xlabel("Wavelength [nm]")
    return fig, axes


compare_observed_by_arm(bundles)
plt.show()

In [ ]:
def compare_collapsed_transmission(bundles):
    fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
    any_product = False
    for b in bundles:
        if b["collapsed"] is None:
            continue
        any_product = True
        c = b["collapsed"]
        wb, vb, eb = binned_series(c["wavelength"], c["spectrum"] - 1.0, c["uncertainty"], max_bins=COLLAPSED_MAX_BINS)
        ax.errorbar(display_wavelength_nm(wb), vb, yerr=eb, fmt="o", ms=2.0, lw=0.55, alpha=0.75, label=f"{b['epoch']} {b['arm']}")
    if not any_product:
        print("No collapsed transmission products found.")
        return None
    ax.axhline(0, color="0.35", lw=0.8, ls="--")
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Transmission - 1")
    ax.set_title("Collapsed transmission products compared")
    ax.legend(loc="best")
    return fig, ax


compare_collapsed_transmission(bundles)
plt.show()

## Line Microscope

The top panel is the phase-ordered residual matrix around a rest wavelength. The dashed curve is the expected planet trail using `Kp * sin(2*pi*phase)` by default. The bottom panel shifts each row by that expected trail and stacks the line window in the planet frame. A real transmission absorption line should prefer a negative feature near 0 km/s in the bottom panel.

In [ ]:
LINE_LIST = [
    {"label": "Ca II K", "rest_A": 3933.663, "arm": "blue"},
    {"label": "Ca II H", "rest_A": 3968.469, "arm": "blue"},
    {"label": "H beta", "rest_A": 4861.333, "arm": "blue"},
    {"label": "Mg I b 5167", "rest_A": 5167.321, "arm": "blue"},
    {"label": "Mg I b 5172", "rest_A": 5172.684, "arm": "blue"},
    {"label": "Mg I b 5183", "rest_A": 5183.604, "arm": "blue"},
    {"label": "Fe II 5169", "rest_A": 5169.033, "arm": "blue"},
    {"label": "Na I D2", "rest_A": 5889.951, "arm": "red"},
    {"label": "Na I D1", "rest_A": 5895.924, "arm": "red"},
    {"label": "H alpha", "rest_A": 6562.800, "arm": "red"},
    {"label": "Li I", "rest_A": 6707.800, "arm": "red"},
]
DEFAULT_LINE_LABELS = ("H beta", "Mg I b 5183", "H alpha")


def line_record(label):
    matches = [line for line in LINE_LIST if line["label"] == label]
    if not matches:
        raise KeyError(label)
    return matches[0]


pd.DataFrame(LINE_LIST)

In [ ]:
def line_window_arrays(bundle, rest_A, window_kms=LINE_WINDOW_KMS):
    wave = np.asarray(bundle["wavelength"], dtype=float)
    velocity = CKMS * (wave / float(rest_A) - 1.0)
    keep = np.isfinite(velocity) & (np.abs(velocity) <= window_kms)
    return velocity[keep], bundle["data"][:, keep], bundle["sigma"][:, keep]


def stack_line_planet_frame(bundle, rest_A, window_kms=LINE_WINDOW_KMS, bin_kms=LINE_BIN_KMS):
    velocity, data, sigma = line_window_arrays(bundle, rest_A, window_kms=window_kms)
    if velocity.size < 4:
        return None
    v_grid = np.arange(-window_kms, window_kms + 0.5 * bin_kms, bin_kms)
    sum_w = np.zeros_like(v_grid, dtype=float)
    sum_yw = np.zeros_like(v_grid, dtype=float)
    for row, sig, phase in zip(data, sigma, bundle["phase"]):
        good = finite_good(row, sig)
        if np.count_nonzero(good) < 4:
            continue
        v_planet_frame = velocity[good] - planet_velocity_kms(phase)
        y = row[good]
        e = sig[good]
        interp_y = np.interp(v_grid, v_planet_frame, y, left=np.nan, right=np.nan)
        interp_e = np.interp(v_grid, v_planet_frame, e, left=np.nan, right=np.nan)
        valid = np.isfinite(interp_y) & np.isfinite(interp_e) & (interp_e > 0) & (interp_e < MASK_SIGMA_THRESHOLD)
        w = np.zeros_like(v_grid, dtype=float)
        w[valid] = 1.0 / np.square(interp_e[valid])
        sum_w += w
        sum_yw += np.where(valid, interp_y * w, 0.0)
    mean = np.divide(sum_yw, sum_w, out=np.full_like(v_grid, np.nan), where=sum_w > 0)
    err = np.divide(1.0, np.sqrt(sum_w), out=np.full_like(v_grid, np.nan), where=sum_w > 0)
    return v_grid, mean, err


def plot_line_microscope(bundle, line):
    rest_A = float(line["rest_A"])
    velocity, data, sigma = line_window_arrays(bundle, rest_A)
    if velocity.size < 4:
        print(f"{bundle['epoch']} {bundle['arm']} does not cover {line['label']} at {rest_A:.3f} A")
        return None
    order = phase_order(bundle)
    phase = bundle["phase"][order]
    mat = np.asarray(data, dtype=float)[order]
    sig = np.asarray(sigma, dtype=float)[order]
    mat = np.where(finite_good(mat, sig), mat, np.nan)
    limits = robust_limits(mat, percentile=99.2, symmetric=True, floor=2e-5)
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 6.2), sharex=False, constrained_layout=True, gridspec_kw={"height_ratios": [2.2, 1.0], "hspace": 0.22})
    im = axes[0].imshow(
        mat,
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=None if limits is None else limits[0],
        vmax=None if limits is None else limits[1],
        extent=[-LINE_WINDOW_KMS, LINE_WINDOW_KMS, len(order) - 0.5, -0.5],
    )
    trail = planet_velocity_kms(phase)
    axes[0].plot(trail, np.arange(len(phase)), color="gold", lw=1.0, ls="--", label="expected planet trail")
    axes[0].axvline(0, color="0.25", lw=0.8, ls=":")
    ticks = np.arange(len(order))
    keep = ticks if len(ticks) <= 14 else np.unique(np.linspace(0, len(ticks) - 1, 12).astype(int))
    axes[0].set_yticks(keep)
    axes[0].set_yticklabels([f"{phase[i]:+.4f}" for i in keep])
    axes[0].set_ylabel("Orbital phase")
    axes[0].legend(loc="upper right")
    cb = fig.colorbar(im, ax=axes[0], pad=0.012, fraction=0.045)
    cb.set_label("Residual flux")

    stacked = stack_line_planet_frame(bundle, rest_A)
    if stacked is None:
        axes[1].text(0.5, 0.5, "No stack available", ha="center", va="center", transform=axes[1].transAxes)
    else:
        v_grid, mean, err = stacked
        axes[1].errorbar(v_grid, mean, yerr=err, fmt="o-", ms=2.0, lw=0.8, color="k", ecolor="0.45")
        axes[1].axhline(0, color="0.35", lw=0.8, ls="--")
        axes[1].axvline(0, color="0.25", lw=0.8, ls=":")
        lim = robust_limits(mean, percentile=98.5, symmetric=True, floor=2e-5)
        if lim:
            axes[1].set_ylim(*lim)
    axes[1].set_xlabel("Velocity around rest wavelength [km/s]")
    axes[1].set_ylabel("Planet-frame stack")
    axes[0].set_title(f"{line['label']} at {rest_A:.3f} A: {bundle['epoch']} {bundle['arm']}")
    return fig, axes

In [ ]:
lines_to_plot = [line_record(label) for label in DEFAULT_LINE_LABELS]
for line in lines_to_plot:
    for b in plot_bundles:
        if b["arm"] != line["arm"]:
            continue
        plot_line_microscope(b, line)
        plt.show()

## SYSREM Products

This summarizes saved SYSREM side products. A missing `U_sysrem.npz` means that particular prepared bundle was not exported with SYSREM enabled.

In [ ]:
def sysrem_summary_rows(bundles):
    rows = []
    for b in bundles:
        z = b.get("sysrem")
        if z is None:
            rows.append({"epoch": b["epoch"], "arm": b["arm"], "has_sysrem": False})
            continue
        row = {"epoch": b["epoch"], "arm": b["arm"], "has_sysrem": True, "files": tuple(z.files)}
        if "basis_counts" in z.files:
            row["basis_counts"] = np.asarray(z["basis_counts"]).tolist()
        if "chunk_names" in z.files:
            row["chunk_names"] = [str(x) for x in np.asarray(z["chunk_names"]).tolist()]
        for name in ("sysrem_delta_stddev", "sysrem_stddev_before", "sysrem_stddev_after"):
            if name in z.files:
                row[name] = np.asarray(z[name]).tolist()
        rows.append(row)
    return rows


sysrem_df = pd.DataFrame(sysrem_summary_rows(bundles))
display(sysrem_df)

## Other KELT-20b Transmission Inputs On Disk

This section just inventories lower-resolution and photometric transmission inputs available in the tree. It does not try to reduce them.

In [ ]:
def inventory_tree(root: Path, max_files=80):
    if not root.exists():
        return pd.DataFrame([{"root": str(root.relative_to(REPO_ROOT)), "exists": False}])
    rows = []
    for path in sorted(root.rglob("*"))[:max_files]:
        rows.append({
            "path": str(path.relative_to(REPO_ROOT)),
            "kind": "dir" if path.is_dir() else "file",
            "size_mb": None if path.is_dir() else round(path.stat().st_size / 1024**2, 3),
        })
    return pd.DataFrame(rows)


display(inventory_tree(LRS_ROOT, max_files=80))
display(inventory_tree(PHOT_ROOT, max_files=80))